In [1]:

# Step 12A — Initialize SHML environment

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cwd = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        p for p in (cwd, *cwd.parents)
        if (p / "src" / "run_pipeline.py").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the SHML repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import evaluate_model
from src.safety import evaluate_recovery_safety

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "working"
    / "default of credit card clients.xls"
)

df = pd.read_excel(DATA_PATH, header=1, engine="xlrd")

target = "default payment next month"

X = df.drop(columns=["ID", target])
y = df[target]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

baseline_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_scaled, y_train)

healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=baseline_model,
    scaler=scaler
)

assert len(X_train) == 21000
assert len(X_val) == 4500
assert len(X_test) == 4500
assert abs(healthy_metrics["f1"] - 0.356983) < 0.0001

print("SHML initialized successfully.")
print("Healthy F1:", round(healthy_metrics["f1"], 6))
print("Healthy recall:", round(healthy_metrics["recall"], 6))
print("Healthy AUC:", round(healthy_metrics["roc_auc"], 6))


SHML initialized successfully.
Healthy F1: 0.356983
Healthy recall: 0.240201
Healthy AUC: 0.713025


In [2]:

# Step 14A — Prepare recovery strategies

from sklearn.linear_model import LinearRegression

FEATURE = "BILL_AMT1"
PREDICTOR = "BILL_AMT2"

training_median = X_train[FEATURE].median()

regression_imputer = LinearRegression()
regression_imputer.fit(
    X_train[[PREDICTOR]],
    X_train[FEATURE]
)

print("Training median:", training_median)
print("Regression imputer initialized.")


Training median: 22225.5
Regression imputer initialized.


In [3]:

# Step 14B — Recovery benefit experiment

results = []

missingness_levels = [0.05, 0.15, 0.30, 0.60]
seeds = range(20)

for seed in seeds:
    for rate in missingness_levels:

        X_corrupted = X_val.copy(deep=True)

        rng = np.random.default_rng(seed)
        n_missing = int(len(X_corrupted) * rate)

        positions = rng.choice(
            len(X_corrupted),
            size=n_missing,
            replace=False
        )

        mask = np.zeros(len(X_corrupted), dtype=bool)
        mask[positions] = True

        X_corrupted.iloc[
            positions,
            X_corrupted.columns.get_loc(FEATURE)
        ] = np.nan

        # Fixed nonadaptive median fallback
        X_median = X_corrupted.copy(deep=True)
        X_median[FEATURE] = X_median[FEATURE].fillna(
            training_median
        )

        # Zero recovery candidate
        X_zero = X_corrupted.copy(deep=True)
        X_zero[FEATURE] = X_zero[FEATURE].fillna(0)

        # Regression recovery candidate
        X_regression = X_corrupted.copy(deep=True)

        if mask.any():
            predictions = regression_imputer.predict(
                X_corrupted.loc[mask, [PREDICTOR]]
            )
            X_regression.loc[mask, FEATURE] = predictions

        candidates = {
            "Fixed Median": X_median,
            "Zero Recovery": X_zero,
            "Regression Recovery": X_regression
        }

        for strategy, X_candidate in candidates.items():

            assert not X_candidate.isna().any().any()

            metrics = evaluate_model(
                X_batch=X_candidate,
                y_true=y_val,
                model=baseline_model,
                scaler=scaler
            )

            results.append({
                "Seed": seed,
                "Missingness": rate,
                "Strategy": strategy,
                "Coverage": 1.0,
                "F1": metrics["f1"],
                "Recall": metrics["recall"],
                "ROC_AUC": metrics["roc_auc"]
            })

        # No-recovery abstention policy:
        # Only score rows with observed BILL_AMT1
        valid_mask = ~mask

        X_observed = X_corrupted.loc[valid_mask]
        y_observed = y_val.loc[valid_mask]

        abstention_metrics = evaluate_model(
            X_batch=X_observed,
            y_true=y_observed,
            model=baseline_model,
            scaler=scaler
        )

        results.append({
            "Seed": seed,
            "Missingness": rate,
            "Strategy": "Abstention",
            "Coverage": valid_mask.mean(),
            "F1": abstention_metrics["f1"],
            "Recall": abstention_metrics["recall"],
            "ROC_AUC": abstention_metrics["roc_auc"]
        })

benefit_df = pd.DataFrame(results)

assert len(benefit_df) == 320

benefit_summary = (
    benefit_df
    .groupby(["Missingness", "Strategy"])
    .agg(
        Evaluations=("Seed", "size"),
        Mean_Coverage=("Coverage", "mean"),
        Mean_F1=("F1", "mean"),
        Mean_Recall=("Recall", "mean"),
        Mean_AUC=("ROC_AUC", "mean")
    )
    .reset_index()
)

display(benefit_summary.round(4))


,Missingness,Strategy,Evaluations,Mean_Coverage,Mean_F1,Mean_Recall,Mean_AUC
0,0.05,Abstention,20,0.95,0.3570,0.2402,0.7131
1,0.05,Fixed Median,20,1.00,0.3593,0.2425,0.7120
2,0.05,Regression Recovery,20,1.00,0.3569,0.2401,0.7130
3,0.05,Zero Recovery,20,1.00,0.3608,0.2441,0.7117
4,0.15,Abstention,20,0.85,0.3575,0.2409,0.7122
5,0.15,Fixed Median,20,1.00,0.3648,0.2479,0.7112
6,0.15,Regression Recovery,20,1.00,0.3573,0.2404,0.7129
7,0.15,Zero Recovery,20,1.00,0.3683,0.2519,0.7107
8,0.30,Abstention,20,0.70,0.3550,0.2387,0.7142
9,0.30,Fixed Median,20,1.00,0.3686,0.2524,0.7096


In [4]:

# Phase F — Step 14C
# Paired recovery benefit relative to fixed median fallback

import numpy as np
import pandas as pd
from scipy import stats

full_coverage = benefit_df[
    benefit_df["Strategy"].isin([
        "Fixed Median",
        "Zero Recovery",
        "Regression Recovery"
    ])
].copy()

paired = full_coverage.pivot(
    index=["Seed", "Missingness"],
    columns="Strategy",
    values=["F1", "Recall", "ROC_AUC"]
)

paired_results = []

for rate in missingness_levels:
    subset = paired.xs(rate, level="Missingness")

    for strategy in [
        "Zero Recovery",
        "Regression Recovery"
    ]:
        for metric in ["F1", "Recall", "ROC_AUC"]:

            differences = (
                subset[(metric, strategy)]
                - subset[(metric, "Fixed Median")]
            ).to_numpy()

            n = len(differences)
            mean_diff = np.mean(differences)

            se = stats.sem(differences)

            if np.isclose(se, 0):
                ci_lower = mean_diff
                ci_upper = mean_diff
            else:
                margin = stats.t.ppf(
                    0.975, df=n - 1
                ) * se

                ci_lower = mean_diff - margin
                ci_upper = mean_diff + margin

            paired_results.append({
                "Missingness": rate,
                "Strategy": strategy,
                "Metric": metric,
                "N_Seeds": n,
                "Mean_Difference": mean_diff,
                "CI_Lower": ci_lower,
                "CI_Upper": ci_upper,
                "Wins": int((differences > 0).sum()),
                "Losses": int((differences < 0).sum()),
                "Ties": int((differences == 0).sum())
            })

paired_benefit_df = pd.DataFrame(paired_results)

assert len(paired_benefit_df) == 24

display(
    paired_benefit_df[
        paired_benefit_df["Metric"].isin([
            "F1", "ROC_AUC"
        ])
    ].round(6)
)


,Missingness,Strategy,Metric,N_Seeds,Mean_Difference,CI_Lower,CI_Upper,Wins,Losses,Ties
0,0.05,Zero Recovery,F1,20,0.001541,0.000655,0.002427,13,6,1
2,0.05,Zero Recovery,ROC_AUC,20,-0.000291,-0.000408,-0.000174,2,18,0
3,0.05,Regression Recovery,F1,20,-0.002364,-0.002969,-0.001760,0,20,0
5,0.05,Regression Recovery,ROC_AUC,20,0.000948,0.000480,0.001415,16,4,0
6,0.15,Zero Recovery,F1,20,0.003559,0.002548,0.004570,19,1,0
8,0.15,Zero Recovery,ROC_AUC,20,-0.000554,-0.000823,-0.000284,3,17,0
9,0.15,Regression Recovery,F1,20,-0.007440,-0.008879,-0.006001,0,20,0
11,0.15,Regression Recovery,ROC_AUC,20,0.001680,0.001167,0.002192,19,1,0
12,0.30,Zero Recovery,F1,20,0.009030,0.007454,0.010605,20,0,0
14,0.30,Zero Recovery,ROC_AUC,20,-0.001002,-0.001361,-0.000644,2,18,0



### Observation F.1 — Paired Recovery Benefit

The experiment compared zero and regression recovery
against a fixed-median fallback under controlled
BILL_AMT1 missingness.

Experiments used 20 paired missingness injection seeds
at 5%, 15%, 30%, and 60% missingness.

#### Findings

Zero imputation achieved higher mean F1 than the
fixed-median fallback at all tested severity levels.

At 60% missingness:

- Zero minus median F1: +0.014666
- 95% CI: [0.012929, 0.016403]
- Positive differences: 20/20 seeds

Regression imputation produced lower F1 but higher
ROC-AUC than the fixed-median fallback.

At 60% missingness:

- Regression minus median F1: -0.025250
- Regression minus median ROC-AUC: +0.007124
- ROC-AUC improvement: 20/20 seeds

#### Interpretation

The findings demonstrate that recovery strategy
choice affects predictive performance.

Zero imputation is favored under the predefined F1
objective, while regression recovery better preserves
ROC-AUC.

The paired confidence intervals quantify variation
across injection seeds with the trained model and
validation dataset held fixed.

#### Limitations

The comparisons use validation data already involved
in experimental development.

The results do not yet establish the performance of
an automated recovery-selection policy on untouched
test data.

Fixed median is a nonadaptive fallback rather than
literal non-intervention.

Improved F1 does not necessarily indicate improved
probability ranking, calibration, or fairness.

The two methods were evaluated on copies of the complete validation features with deliberately masked `BILL_AMT1` values.


In [5]:

# Step 14D — Freeze SHML recovery policy configuration

SHML_POLICY = {
    "feature": "BILL_AMT1",
    "primary_metric": "f1",
    "classification_threshold": 0.50,
    "preferred_strategy": "Zero Recovery",
    "fallback_strategy": "Fixed Median",
    "max_f1_loss": 0.10,
    "max_recall_loss": 0.10,
    "monitor_metric": "roc_auc",
    "policy_source": "validation experiments",
    "status": "frozen for final evaluation"
}

# Verify that the preferred strategy outperformed
# the fallback on mean F1 across tested severities.

f1_comparisons = paired_benefit_df[
    (paired_benefit_df["Strategy"] == "Zero Recovery")
    & (paired_benefit_df["Metric"] == "F1")
]

assert len(f1_comparisons) == 4
assert (f1_comparisons["Mean_Difference"] > 0).all()

print("Recovery policy configuration defined.")
print("Preferred:", SHML_POLICY["preferred_strategy"])
print("Fallback:", SHML_POLICY["fallback_strategy"])

display(pd.DataFrame(
    SHML_POLICY.items(),
    columns=["Setting", "Value"]
))


Recovery policy configuration defined.
Preferred: Zero Recovery
Fallback: Fixed Median


,Setting,Value
0,feature,BILL_AMT1
1,primary_metric,f1
2,classification_threshold,0.5
3,preferred_strategy,Zero Recovery
4,fallback_strategy,Fixed Median
5,max_f1_loss,0.1
6,max_recall_loss,0.1
7,monitor_metric,roc_auc
8,policy_source,validation experiments
9,status,frozen for final evaluation
